In [5]:
from pathlib import Path

from IPython.display import display

import src.phase3.curation as phase3_curation
from src.phase3.curation import (
    build_phase3_all_experiment_specs,
    build_phase3_combined_experiment_specs,
    build_phase3_individual_experiment_specs,
    prepare_phase3_experiment_dataset,
)
from src.curation_evaluation import (
    run_phase3_experiment,
    select_best_phase3_individual_options,
    select_best_phase3_spec,
    summarize_phase3_experiment_specs,
)

FORCE_REBUILD_DATASETS = False
FORCE_SCORE = False
FORCE_TRAIN = False
PHASE3_CURATION_DATA_DIR = Path("phase3test")
PHASE3_EVALUATION_RESULTS_ROOT = Path("phase3_test")
RESULTS_CSV_PATH = Path("results") / "phase3_sweep_results.csv"

phase3_curation.PHASE3_CURATION_DATA_DIR = PHASE3_CURATION_DATA_DIR

print(f"CSV output dir: data/{PHASE3_CURATION_DATA_DIR}")
print(f"Training output root: results/<architecture>/{PHASE3_EVALUATION_RESULTS_ROOT}")

CSV output dir: data/phase3test
Training output root: results/<architecture>/phase3_test


## 1. Definir experimentos individuales

Se prueban por separado `top%`, `dedup` y `quality` sobre cada dataset de partida de fase 2.

In [6]:
individual_specs = build_phase3_individual_experiment_specs()

print(f"Individual datasets: {len(individual_specs)}")
for spec in individual_specs:
    print(f"- {spec.descriptor}: {spec.output_csv}")

Individual datasets: 18
- train_conf040_top60: phase3test\phase3_train_conf040_top60.csv
- train_conf040_top70: phase3test\phase3_train_conf040_top70.csv
- train_conf040_top80: phase3test\phase3_train_conf040_top80.csv
- train_conf040_dedup_config: phase3test\phase3_train_conf040_dedup_config.csv
- train_conf040_dedup_p90_10: phase3test\phase3_train_conf040_dedup_p90_10.csv
- train_conf040_dedup_p75_25: phase3test\phase3_train_conf040_dedup_p75_25.csv
- train_conf040_quality_config: phase3test\phase3_train_conf040_quality_config.csv
- train_conf040_quality_p10: phase3test\phase3_train_conf040_quality_p10.csv
- train_conf040_quality_p25: phase3test\phase3_train_conf040_quality_p25.csv
- conf060_top60: phase3test\phase3_conf060_top60.csv
- conf060_top70: phase3test\phase3_conf060_top70.csv
- conf060_top80: phase3test\phase3_conf060_top80.csv
- conf060_dedup_config: phase3test\phase3_conf060_dedup_config.csv
- conf060_dedup_p90_10: phase3test\phase3_conf060_dedup_p90_10.csv
- conf060_dedu

## 2. Crear CSVs individuales

Los CSVs se guardan en `data/phase3test`. Si ya existen y `FORCE_REBUILD_DATASETS=False`, se reutilizan.

In [7]:
prepared_individual = []

for index, spec in enumerate(individual_specs, start=1):
    print(f"[{index}/{len(individual_specs)}] Preparing {spec.descriptor}")
    result = prepare_phase3_experiment_dataset(
        spec,
        force_rebuild=FORCE_REBUILD_DATASETS,
        force_score=FORCE_SCORE,
    )
    prepared_individual.append(result)
    print(f"    output_csv={result['output_csv']} | reused={result['reused']}")

[1/18] Preparing train_conf040_top60
    output_csv=phase3test\phase3_train_conf040_top60.csv | reused=True
[2/18] Preparing train_conf040_top70
    output_csv=phase3test\phase3_train_conf040_top70.csv | reused=True
[3/18] Preparing train_conf040_top80
    output_csv=phase3test\phase3_train_conf040_top80.csv | reused=True
[4/18] Preparing train_conf040_dedup_config
    output_csv=phase3test\phase3_train_conf040_dedup_config.csv | reused=True
[5/18] Preparing train_conf040_dedup_p90_10
    output_csv=phase3test\phase3_train_conf040_dedup_p90_10.csv | reused=True
[6/18] Preparing train_conf040_dedup_p75_25
    output_csv=phase3test\phase3_train_conf040_dedup_p75_25.csv | reused=True
[7/18] Preparing train_conf040_quality_config
    output_csv=phase3test\phase3_train_conf040_quality_config.csv | reused=True
[8/18] Preparing train_conf040_quality_p10
    output_csv=phase3test\phase3_train_conf040_quality_p10.csv | reused=True
[9/18] Preparing train_conf040_quality_p25
    output_csv=phase3

## 3. Entrenar individuales y mostrar metricas

Cada experimento usa 2 seeds. Los plots se guardan por el entrenamiento, pero no se muestran.

In [8]:
individual_metric_tables = {}

for index, spec in enumerate(individual_specs, start=1):
    print("=" * 100)
    print(f"[{index}/{len(individual_specs)}] Training {spec.descriptor}")
    metrics_df = run_phase3_experiment(
        spec,
        force_rebuild_dataset=False,
        force_score=FORCE_SCORE,
        force_train=FORCE_TRAIN,
        results_root=PHASE3_EVALUATION_RESULTS_ROOT,
    )
    individual_metric_tables[spec.descriptor] = metrics_df

[1/18] Training train_conf040_top60
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top60\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top60\seed_2\best_baseline_model.pth
[2/18] Training train_conf040_top70
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top70\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top70\seed_2\best_baseline_model.pth
[3/18] Training train_conf040_top80
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top80\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_c

## 4. Resumen individual y seleccion de mejores parametros

La seleccion usa `macro_f1`; si la diferencia es menor que `0.005`, desempata por `mcc`.

In [9]:
individual_summary = summarize_phase3_experiment_specs(
    individual_specs,
    results_root=PHASE3_EVALUATION_RESULTS_ROOT,
)

best_options = select_best_phase3_individual_options(individual_summary)
print("Best top fraction:", best_options["best_top_fraction"])
print("Best dedup mode:", best_options["best_dedup_mode"])
print("Best quality mode:", best_options["best_quality_mode"])

Best top fraction: 0.6
Best dedup mode: p75_25
Best quality mode: config


## 5. Definir combinaciones finales

Con los mejores valores encontrados, se prueban solo combinaciones de 2 o 3 condiciones: `top+dedup`, `top+quality`, `dedup+quality`, `top+dedup+quality`.

In [10]:
combined_specs = build_phase3_combined_experiment_specs(
    best_top_fraction=best_options["best_top_fraction"],
    best_dedup_mode=best_options["best_dedup_mode"],
    best_quality_mode=best_options["best_quality_mode"],
)

print(f"Combined datasets: {len(combined_specs)}")
for spec in combined_specs:
    print(f"- {spec.descriptor}: {spec.output_csv}")

Combined datasets: 8
- train_conf040_top60_dedup_p75_25: phase3test\phase3_train_conf040_top60_dedup_p75_25.csv
- train_conf040_top60_quality_config: phase3test\phase3_train_conf040_top60_quality_config.csv
- train_conf040_dedup_p75_25_quality_config: phase3test\phase3_train_conf040_dedup_p75_25_quality_config.csv
- train_conf040_top60_dedup_p75_25_quality_config: phase3test\phase3_train_conf040_top60_dedup_p75_25_quality_config.csv
- conf060_top60_dedup_p75_25: phase3test\phase3_conf060_top60_dedup_p75_25.csv
- conf060_top60_quality_config: phase3test\phase3_conf060_top60_quality_config.csv
- conf060_dedup_p75_25_quality_config: phase3test\phase3_conf060_dedup_p75_25_quality_config.csv
- conf060_top60_dedup_p75_25_quality_config: phase3test\phase3_conf060_top60_dedup_p75_25_quality_config.csv


## 6. Crear CSVs combinados

In [11]:
prepared_combined = []

for index, spec in enumerate(combined_specs, start=1):
    print(f"[{index}/{len(combined_specs)}] Preparing {spec.descriptor}")
    result = prepare_phase3_experiment_dataset(
        spec,
        force_rebuild=FORCE_REBUILD_DATASETS,
        force_score=FORCE_SCORE,
    )
    prepared_combined.append(result)
    print(f"    output_csv={result['output_csv']} | reused={result['reused']}")

[1/8] Preparing train_conf040_top60_dedup_p75_25
    output_csv=phase3test\phase3_train_conf040_top60_dedup_p75_25.csv | reused=True
[2/8] Preparing train_conf040_top60_quality_config
    output_csv=phase3test\phase3_train_conf040_top60_quality_config.csv | reused=True
[3/8] Preparing train_conf040_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_train_conf040_dedup_p75_25_quality_config.csv | reused=True
[4/8] Preparing train_conf040_top60_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_train_conf040_top60_dedup_p75_25_quality_config.csv | reused=True
[5/8] Preparing conf060_top60_dedup_p75_25
    output_csv=phase3test\phase3_conf060_top60_dedup_p75_25.csv | reused=True
[6/8] Preparing conf060_top60_quality_config
    output_csv=phase3test\phase3_conf060_top60_quality_config.csv | reused=True
[7/8] Preparing conf060_dedup_p75_25_quality_config
    output_csv=phase3test\phase3_conf060_dedup_p75_25_quality_config.csv | reused=True
[8/8] Preparing conf060_top60_d

## 7. Entrenar combinados y mostrar metricas

In [12]:
combined_metric_tables = {}

for index, spec in enumerate(combined_specs, start=1):
    print("=" * 100)
    print(f"[{index}/{len(combined_specs)}] Training {spec.descriptor}")
    metrics_df = run_phase3_experiment(
        spec,
        force_rebuild_dataset=False,
        force_score=FORCE_SCORE,
        force_train=FORCE_TRAIN,
        results_root=PHASE3_EVALUATION_RESULTS_ROOT,
    )
    combined_metric_tables[spec.descriptor] = metrics_df

[1/8] Training train_conf040_top60_dedup_p75_25
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top60_dedup_p75_25\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top60_dedup_p75_25\seed_2\best_baseline_model.pth
[2/8] Training train_conf040_top60_quality_config
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top60_quality_config\seed_1\best_baseline_model.pth
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_top60_quality_config\seed_2\best_baseline_model.pth
[3/8] Training train_conf040_dedup_p75_25_quality_config
Loading model located at: C:\Users\luis\Documents\TFG - Data-Centric AI\results\EfficientNet\phase3_test\train_conf040_dedup_p75_25_quality_config\seed_1\best_baseli

## 8. Tabla final general

Tabla sin clases, ordenada por `macro_f1` y `mcc`.

In [13]:
all_specs = build_phase3_all_experiment_specs(
    best_top_fraction=best_options["best_top_fraction"],
    best_dedup_mode=best_options["best_dedup_mode"],
    best_quality_mode=best_options["best_quality_mode"],
)

final_summary = summarize_phase3_experiment_specs(
    all_specs,
    results_root=PHASE3_EVALUATION_RESULTS_ROOT,
)
best_final = select_best_phase3_spec(final_summary)
results_for_export = final_summary.assign(
    is_best=lambda df: df["descriptor"].eq(best_final["descriptor"])
)
RESULTS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
results_for_export.to_csv(RESULTS_CSV_PATH, index=False)

display(results_for_export)
print("Best final experiment:")
display(best_final.to_frame().T)
print(f"Saved sweep results to {RESULTS_CSV_PATH}")

,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,macro_f1,mcc,accuracy,precision,recall,is_best
0,conf060_dedup_config,conf060,dedup,NaN,config,NaN,phase3test\phase3_conf060_dedup_config.csv,0.584553,0.282075,0.547059,0.596198,0.589808,False
1,conf060_dedup_p75_25,conf060,dedup,NaN,p75_25,NaN,phase3test\phase3_conf060_dedup_p75_25.csv,0.584284,0.287859,0.561765,0.585882,0.592226,True
2,train_conf040_dedup_p75_25,train_conf040,dedup,NaN,p75_25,NaN,phase3test\phase3_train_conf040_dedup_p75_25.csv,0.579952,0.276157,0.556618,0.584390,0.583835,False
3,conf060_quality_config,conf060,quality,NaN,NaN,config,phase3test\phase3_conf060_quality_config.csv,0.579297,0.281073,0.564706,0.580212,0.584300,False
4,conf060_top80,conf060,top,0.8,NaN,NaN,phase3test\phase3_conf060_top80.csv,0.574968,0.268520,0.536029,0.566525,0.600145,False
5,conf060_dedup_p90_10,conf060,dedup,NaN,p90_10,NaN,phase3test\phase3_conf060_dedup_p90_10.csv,0.574805,0.273570,0.551471,0.576944,0.582799,False
6,conf060_dedup_p75_25_quality_config,conf060,dedup+quality,NaN,p75_25,config,phase3test\phase3_conf060_dedup_p75_25_quality...,0.573538,0.265530,0.543382,0.570316,0.588657,False
7,train_conf040_top60_dedup_p75_25_quality_config,train_conf040,top+dedup+quality,0.6,p75_25,config,phase3test\phase3_train_conf040_top60_dedup_p7...,0.572103,0.259095,0.533824,0.587151,0.572611,False
8,train_conf040_top60,train_conf040,top,0.6,NaN,NaN,phase3test\phase3_train_conf040_top60.csv,0.571595,0.270439,0.547059,0.574375,0.579540,False
9,train_conf040_dedup_p75_25_quality_config,train_conf040,dedup+quality,NaN,p75_25,config,phase3test\phase3_train_conf040_dedup_p75_25_q...,0.570801,0.257643,0.534559,0.588419,0.569069,False


Best final experiment:


,descriptor,source,operations,top_fraction,dedup_mode,quality_mode,output_csv,macro_f1,mcc,accuracy,precision,recall
1,conf060_dedup_p75_25,conf060,dedup,NaN,p75_25,NaN,phase3test\phase3_conf060_dedup_p75_25.csv,0.584284,0.287859,0.561765,0.585882,0.592226


Saved sweep results to results\phase3_sweep_results.csv
